# 01 – Análise exploratória da MetroPT-3

**Objetivo:** entender a estrutura, a qualidade e o comportamento dos sinais antes de construir o pipeline Bronze/Silver/Gold.

Perguntas a responder aqui:
1. Quais colunas existem e quais são analógicas × digitais?
2. Há nulos, duplicatas ou timestamps fora de ordem?
3. Qual o intervalo de amostragem e há lacunas no tempo?
4. Os ranges dos sensores fazem sentido físico?
5. Como os sinais se comportam perto das falhas do relatório?

*Fonte: Davari et al., MetroPT-3 Dataset, UCI ML Repository (DOI 10.24432/C5VW3R), CC BY 4.0.*

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from config import FAILURE_REPORT, find_raw_csv

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)

## 1. Carga e estrutura

In [ ]:
raw_path = find_raw_csv()
df = pd.read_csv(raw_path)
print(raw_path.name, "->", df.shape)
df.head()

In [ ]:
# Remove coluna de índice sem nome (se existir) e identifica a coluna de tempo
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
time_col = next(c for c in df.columns if "time" in c.lower())
df[time_col] = pd.to_datetime(df[time_col])
fora_de_ordem = not df[time_col].is_monotonic_increasing
df = df.sort_values(time_col).reset_index(drop=True)

# Digital = no máximo 2 valores distintos; analógico = o restante numérico
numeric = [c for c in df.columns if c != time_col and pd.api.types.is_numeric_dtype(df[c])]
digital = [c for c in numeric if df[c].nunique() <= 2]
analog = [c for c in numeric if c not in digital]
print("Analógicos:", analog)
print("Digitais:  ", digital)
df.dtypes

## 2. Qualidade dos dados

In [ ]:
quality = pd.DataFrame({
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(3),
    "valores distintos": df.nunique(),
})
print("Linhas duplicadas:", df.duplicated().sum())
print("Timestamps duplicados:", df[time_col].duplicated().sum())
print("Timestamps estavam fora de ordem no arquivo:", fora_de_ordem)
quality

In [ ]:
df[analog].describe().T

## 3. Cobertura e amostragem no tempo

In [ ]:
delta = df[time_col].diff().dropna()
print("Período:", df[time_col].min(), "->", df[time_col].max())
print("Intervalo mais comum entre leituras:", delta.mode().iloc[0])
print(delta.describe())
gaps = delta[delta > delta.mode().iloc[0] * 5]
print("Lacunas (> 5x o intervalo típico):", len(gaps))
gaps.sort_values(ascending=False).head(10)

In [ ]:
daily = df.set_index(time_col).resample("D").size()
daily.plot(kind="bar", title="Leituras por dia", figsize=(14, 3))
plt.tight_layout()
plt.show()

## 4. Comportamento dos sinais

In [ ]:
sample = df.set_index(time_col)[analog].resample("1h").mean()
sample.plot(subplots=True, figsize=(14, 2 * len(analog)), sharex=True, title="Média horária dos sinais analógicos")
plt.tight_layout()
plt.show()

In [ ]:
df[analog].hist(bins=50, figsize=(14, 8))
plt.suptitle("Distribuição dos sinais analógicos")
plt.tight_layout()
plt.show()

## 5. Falhas do relatório de manutenção

Preencha `data/external/failure_reports.csv` com a tabela de falhas da página da base na UCI
(colunas: `failure_id,start_time,end_time,failure_type,severity`) e confira os dados antes de usar.
A célula abaixo só roda se o arquivo tiver linhas.

In [ ]:
failures = pd.read_csv(FAILURE_REPORT, parse_dates=["start_time", "end_time"])
print(f"{len(failures)} falhas no relatório")
failures

In [ ]:
if len(failures) == 0:
    print("Preencha data/external/failure_reports.csv para continuar.")
else:
    hourly = df.set_index(time_col)[analog].resample("1h").mean()
    for col in analog[:4]:
        ax = hourly[col].plot(figsize=(14, 3), title=col)
        for _, f in failures.iterrows():
            ax.axvspan(f["start_time"], f["end_time"], color="red", alpha=0.3)
        plt.tight_layout()
        plt.show()

    # Fração de linhas dentro de janelas de falha (desbalanceamento)
    in_failure = pd.Series(False, index=df.index)
    for _, f in failures.iterrows():
        in_failure |= (df[time_col] >= f["start_time"]) & (df[time_col] <= f["end_time"])
    print(f"Linhas em janela de falha: {in_failure.sum()} ({in_failure.mean():.2%})")

## 6. Conclusões da exploração (preencher)

- Nulos/duplicatas encontrados:
- Lacunas no tempo:
- Sensores com valores fisicamente suspeitos:
- Número de falhas e duração:
- Decisões para a camada Silver:
- Decisões para a rotulagem (Gold):